# Notebook 1 — Data Preprocessing
**Adaptive Escalation Framework for Intent-Aware AI-Native Network Slicing in 6G Healthcare Networks**

Steps covered:
1. Dataset generation / loading
2. Cleaning & missing-value handling
3. Feature engineering
4. Traffic visualisation
5. Feature scaling
6. Train / test split
7. Save scaler

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from modules.preprocessing import (
    generate_dataset, clean_data, engineer_features,
    scale_features, build_sequences, train_test_split_ts, FEATURE_COLS
)
from config import DATA_PATH, SCALER_PATH, SEQ_LEN

print('Libraries loaded.')

## 1. Generate & Load Dataset

In [ ]:
df_raw = generate_dataset()
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_raw.to_csv(DATA_PATH, index=False)
print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

## 2. Cleaning & Missing Value Handling

In [ ]:
print('Missing values before cleaning:')
print(df_raw.isnull().sum())

df_clean = clean_data(df_raw)
print('\nMissing values after cleaning:')
print(df_clean.isnull().sum())
print(f'\nDuplicates removed: {len(df_raw) - len(df_clean)}')
df_clean.describe()

## 3. Feature Engineering

In [ ]:
df_feat = engineer_features(df_clean)
print('New features added:')
new_cols = [c for c in df_feat.columns if c not in df_clean.columns]
print(new_cols)
df_feat[new_cols].head()

## 4. Traffic Visualisation

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Traffic (Mbps)', 'Latency (ms)', 'Anomaly Labels'))

fig.add_trace(go.Scatter(x=df_feat['timestamp'], y=df_feat['traffic_mbps'],
                         line=dict(color='#00B4D8'), name='Traffic'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_feat['timestamp'], y=df_feat['latency_ms'],
                         line=dict(color='#F77F00'), name='Latency'), row=2, col=1)
fig.add_trace(go.Bar(x=df_feat['timestamp'], y=df_feat['anomaly_label'],
                     marker_color='#E63946', name='Anomaly'), row=3, col=1)

fig.update_layout(template='plotly_dark', height=600,
                  title='Healthcare Network Traffic Overview')
fig.show()

# Device type distribution
px.pie(df_feat, names='device_type', title='Device Type Distribution',
       template='plotly_dark').show()

## 5. Feature Scaling & Save Scaler

In [ ]:
df_scaled, scaler = scale_features(df_feat, fit=True)
print(f'Scaler saved to: {SCALER_PATH}')
print('Scaled feature range (should be [0,1]):')
print(df_scaled[FEATURE_COLS].describe().loc[['min','max']])

## 6. Build Sequences & Train/Test Split

In [ ]:
traffic_series = df_scaled['traffic_mbps'].values
X, y = build_sequences(traffic_series, seq_len=SEQ_LEN)
X_tr, X_te, y_tr, y_te = train_test_split_ts(X, y)

print(f'X_train: {X_tr.shape}  y_train: {y_tr.shape}')
print(f'X_test : {X_te.shape}  y_test : {y_te.shape}')

# Save splits for downstream notebooks
import numpy as np
np.save('../models/X_tr.npy', X_tr)
np.save('../models/X_te.npy', X_te)
np.save('../models/y_tr.npy', y_tr)
np.save('../models/y_te.npy', y_te)
print('Splits saved to models/.')

## 7. Correlation Heatmap

In [ ]:
corr = df_scaled[FEATURE_COLS].corr()
fig_corr = px.imshow(corr, text_auto='.2f', aspect='auto',
                     color_continuous_scale='RdBu_r',
                     title='Feature Correlation Matrix',
                     template='plotly_dark')
fig_corr.show()
print('Notebook 1 complete.')